# RAG Document Assistant

This notebook builds and evaluates a simple retrieval-augmented generation (RAG) assistant over the course PDFs in `data/raw`. Run all cells from top to bottom.

## 2.1 Load & Inspect

Each extracted page keeps its original PDF filename and one-based page number. Pages with no extractable text are retained in the inspection report but are not sent to the chunker.

In [1]:
from pathlib import Path
import json
import os
import re

import chromadb
import ollama
import pandas as pd
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

# Resolve paths from either the project root or the notebooks directory.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
VECTOR_DIR = PROJECT_ROOT / "backend" / "data" / "vector_store"
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
COLLECTION_NAME = "course_documents"
LLM_NAME = os.getenv("LLM_MODEL", "llama3.2")
RETRIEVAL_K = 6
RELEVANCE_DISTANCE_THRESHOLD = 0.62

def clean_text(text):
    """Normalize whitespace while keeping the page content readable."""
    return re.sub(r"\s+", " ", text or "").strip()

pages = []
report_rows = []
for pdf_path in sorted(RAW_DIR.glob("*.pdf")):
    empty_pages = 0
    try:
        reader = PdfReader(str(pdf_path))
        for page_number, page in enumerate(reader.pages, start=1):
            text = clean_text(page.extract_text())
            if not text:
                empty_pages += 1
            pages.append({
                "source": pdf_path.name,
                "page": page_number,
                "text": text,
            })
        status = "OK"
        page_count = len(reader.pages)
    except Exception as error:
        status = f"FAILED: {type(error).__name__}: {error}"
        page_count = 0
    report_rows.append({
        "file": pdf_path.name,
        "pages": page_count,
        "empty pages": empty_pages,
        "status": status,
    })

pdf_report = pd.DataFrame(report_rows)
print(pdf_report.to_string(index=False))
print(f"\nTotal files: {len(pdf_report)}")
print(f"Total pages: {pdf_report['pages'].sum()}")
print(f"Empty pages: {pdf_report['empty pages'].sum()}")
print(f"Failed files: {(pdf_report['status'] != 'OK').sum()}")

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/BaseFont': '/Helvetica', '/Type': '/Font', '/Name': '/R10', '/FontDescriptor': IndirectObject(368, 0, 1783593753504), '/FirstChar': 32, '/LastChar': 255, '/Widths': [278, 278, 355, 556, 556, 889, 667, 221, 333, 333, 389, 584, 278, 584, 278, 278, 556, 556, 556, 556, 556, 556, 556, 556, 556, 556, 278, 278, 584, 584, 584, 556, 1015, 667, 667, 722, 722, 667, 611, 778, 722, 278, 500, 667, 556, 833, 722, 778, 667, 778, 722, 667, 611, 722, 667, 944, 667, 667, 611, 278, 278, 278, 469, 556, 222, 556, 556, 500, 556, 556, 278, 556, 556, 222, 222, 500, 222, 833, 556, 556, 556, 556, 333, 500, 278, 556, 500, 722, 500, 500, 500, 334, 260, 334, 584, 278, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 944, 1000, 278, 278, 278, 278, 278, 278, 278, 333, 556, 556, 556, 556, 260, 556, 333, 737, 370, 556, 584, 333, 737, 333

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/BaseFont': '/FDDJRZ+Helvetica', '/Type': '/Font', '/Name': '/R10', '/FontDescriptor': IndirectObject(392, 0, 1783593753504), '/FirstChar': 32, '/LastChar': 251, '/Widths': [278, 278, 355, 556, 556, 889, 667, 221, 333, 333, 389, 584, 278, 333, 278, 278, 556, 556, 556, 556, 556, 556, 556, 556, 556, 556, 278, 278, 584, 584, 584, 556, 1015, 667, 667, 722, 722, 667, 611, 778, 722, 278, 500, 667, 556, 833, 722, 778, 667, 778, 722, 667, 611, 722, 667, 944, 667, 667, 611, 278, 278, 278, 469, 556, 222, 556, 556, 500, 556, 556, 278, 556, 556, 222, 222, 500, 222, 833, 556, 556, 556, 556, 333, 500, 278, 556, 500, 722, 500, 500, 500, 334, 260, 334, 584, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 333, 556, 556, 167, 556, 556, 556, 556, 191, 333, 556, 333, 333, 50

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/BaseFont': '/Helvetica', '/Type': '/Font', '/Name': '/R10', '/FontDescriptor': IndirectObject(368, 0, 1783593754720), '/FirstChar': 32, '/LastChar': 255, '/Widths': [278, 278, 355, 556, 556, 889, 667, 221, 333, 333, 389, 584, 278, 584, 278, 278, 556, 556, 556, 556, 556, 556, 556, 556, 556, 556, 278, 278, 584, 584, 584, 556, 1015, 667, 667, 722, 722, 667, 611, 778, 722, 278, 500, 667, 556, 833, 722, 778, 667, 778, 722, 667, 611, 722, 667, 944, 667, 667, 611, 278, 278, 278, 469, 556, 222, 556, 556, 500, 556, 556, 278, 556, 556, 222, 222, 500, 222, 833, 556, 556, 556, 556, 333, 500, 278, 556, 500, 722, 500, 500, 500, 334, 260, 334, 584, 278, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 333, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 944, 1000, 278, 278, 278, 278, 278, 278, 278, 333, 556, 556, 556, 556, 260, 556, 333, 737, 370, 556, 584, 333, 737, 333

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/BaseFont': '/FDDJRZ+Helvetica', '/Type': '/Font', '/Name': '/R10', '/FontDescriptor': IndirectObject(392, 0, 1783593754720), '/FirstChar': 32, '/LastChar': 251, '/Widths': [278, 278, 355, 556, 556, 889, 667, 221, 333, 333, 389, 584, 278, 333, 278, 278, 556, 556, 556, 556, 556, 556, 556, 556, 556, 556, 278, 278, 584, 584, 584, 556, 1015, 667, 667, 722, 722, 667, 611, 778, 722, 278, 500, 667, 556, 833, 722, 778, 667, 778, 722, 667, 611, 722, 667, 944, 667, 667, 611, 278, 278, 278, 469, 556, 222, 556, 556, 500, 556, 556, 278, 556, 556, 222, 222, 500, 222, 833, 556, 556, 556, 556, 333, 500, 278, 556, 500, 722, 500, 500, 500, 334, 260, 334, 584, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 278, 333, 556, 556, 167, 556, 556, 556, 556, 191, 333, 556, 333, 333, 50

                            file  pages  empty pages status
-Mid term-Assembly  Language.pdf      2            0     OK
            Agents in Action.pdf     33            0     OK
                        CH 7.pdf     25            0     OK
                chapter1 (1).pdf     39            0     OK
                    chapter1.pdf     39            0     OK
                chapter2 (1).pdf     46            0     OK
                    chapter2.pdf     46            0     OK
                    chapter3.pdf     33            0     OK
                    chapter5.pdf     30            0     OK
      Density Estimation (1).pdf     12            0     OK
          Density Estimation.pdf     12            0     OK
          Introduction to AI.pdf     29            0     OK
                          L1.pdf     20            0     OK
                          L2.pdf     30            0     OK
                          L3.pdf     56            0     OK
                          L4.pdf     36 

The inspection cell reports **45 PDF files** and their page totals when this notebook is run. A failure is any file whose status is not `OK`; empty pages are counted separately. Failed files are skipped automatically, while successfully extracted non-empty pages continue through the pipeline.

## 2.2 Chunking

Consecutive non-empty pages from the same PDF are merged before chunking. Chunks are about 1000 characters with 200 characters of overlap, which gives slide-like documents enough surrounding context while keeping retrieval focused. Source filename and page range metadata are retained.


In [2]:
def merge_consecutive_pages(page_rows):
    """Join consecutive non-empty pages from each PDF before chunking."""
    merged = []
    current = None
    for row in page_rows:
        if not row["text"]:
            continue
        if current is None or current["source"] != row["source"] or current["page_end"] + 1 != row["page"]:
            if current is not None:
                merged.append(current)
            current = {
                "source": row["source"],
                "page_start": row["page"],
                "page_end": row["page"],
                "text": f"[Page {row['page']}] {row['text']}",
            }
        else:
            current["page_end"] = row["page"]
            current["text"] += f" [Page {row['page']}] {row['text']}"
    if current is not None:
        merged.append(current)
    return merged


def make_chunks(page_rows, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Create fixed-size overlapping chunks from merged page runs."""
    chunks = []
    step = chunk_size - overlap
    for document in merge_consecutive_pages(page_rows):
        text = document["text"]
        for start in range(0, len(text), step):
            chunk_text = text[start:start + chunk_size]
            if chunk_text:
                chunks.append({
                    "text": chunk_text,
                    "metadata": {
                        "source": document["source"],
                        "page": int(re.findall(r"\[Page (\d+)\]", chunk_text)[0]) if re.findall(r"\[Page (\d+)\]", chunk_text) else document["page_start"],
                        "page_end": int(re.findall(r"\[Page (\d+)\]", chunk_text)[-1]) if re.findall(r"\[Page (\d+)\]", chunk_text) else document["page_end"],
                    },
                })
            if start + chunk_size >= len(text):
                break
    return chunks

chunks = make_chunks(pages)
print(f"Created {len(chunks)} chunks from merged page runs.")
print(chunks[0] if chunks else "No chunks were created.")


Created 413 chunks from merged page runs.
{'text': '[Page 1] 1 Academic Semester Fall 2025-2026 Academic level 3 Course Code CS312 Exam Midterm Course Title Assembly Language Time Allowed One Hour Total Points 25 Course Instructors Dr Adel Yehia Ezzat- Dr Ahmed Gaber- Dr Rabab Hamed Question 1: (MCQ) (5 points) 1.The program that converts assembly language to machine language is: a. compiler b. assembler c. linker d. loader 2. The 8086 processor has a. 16– bit address bus and 16-bit data bus b. 12– bit address bus and 16-bit data bus c. 20– bit address bus and 8-bit data bus d. 20– bit address bus and 16-bit data bus 3. The instructions and data are stored in memory in a. binary form b. hexadecimal form c. C/C++ form d. none of the above 4. The zero flag is set to 1 when a. Result is non zero b. result is negative c. Result is zero d. Result is positive [Page 2] 2 Question 2: (10 points) An 8086 processor has three registers AL= 7 , AH= 5 , BL=8 and the carry flag C= 1 Show the content

## 2.3 Embeddings & Vector Store

The `all-MiniLM-L6-v2` sentence-transformer creates compact semantic embeddings. Chroma persists the embeddings and metadata in `backend/data/vector_store`; its HNSW distance is configured for cosine similarity. Stable chunk IDs make rerunning the notebook reproducible.


In [3]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
chroma_client = chromadb.PersistentClient(path=str(VECTOR_DIR))

# Rebuild this named collection so Restart & Run All never leaves stale chunks.
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    configuration={"hnsw": {"space": "cosine"}},
)

chunk_texts = [item["text"] for item in chunks]
chunk_metadata = [item["metadata"] for item in chunks]
chunk_ids = [f"chunk-{index:06d}" for index in range(len(chunks))]
if chunk_texts:
    chunk_embeddings = embedding_model.encode(
        chunk_texts,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).tolist()
    collection.add(
        ids=chunk_ids,
        documents=chunk_texts,
        metadatas=chunk_metadata,
        embeddings=chunk_embeddings,
    )
print(f"Stored {collection.count()} chunks in {VECTOR_DIR} using cosine similarity.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Stored 413 chunks in C:\rag-assistant-project\backend\data\vector_store using cosine similarity.


## 2.4 Retrieval & Prompting

Retrieval returns the six closest chunks and their source metadata. The prompt allows partial answers supported by context, prefixes every chunk with its source filename and page range, and says `I don't know based on the documents` only when no relevant evidence is present.


In [4]:
def retrieve(question, k=RETRIEVAL_K):
    """Return only chunks within the cosine-distance relevance threshold."""
    query_embedding = embedding_model.encode([question], normalize_embeddings=True).tolist()
    result = collection.query(query_embeddings=query_embedding, n_results=k, include=["documents", "metadatas", "distances"])
    return [
        {"text": text, "metadata": metadata, "distance": distance}
        for text, metadata, distance in zip(result["documents"][0], result["metadatas"][0], result["distances"][0])
        if distance <= RELEVANCE_DISTANCE_THRESHOLD
    ]

def raw_distances(question, k=RETRIEVAL_K):
    """Return unfiltered distances for threshold selection diagnostics."""
    query_embedding = embedding_model.encode([question], normalize_embeddings=True).tolist()
    result = collection.query(query_embeddings=query_embedding, n_results=k, include=["distances"])
    return result["distances"][0]

PROMPT_TEMPLATE = """Use the context below to answer the question. If it contains partial information, answer with what it supports and cite it. Only say \"I don't know based on the documents\" if the context is unrelated. Never use outside knowledge.

Example:
Context: Source filename: lecture.pdf | page 2
An assembler converts assembly language into machine language.
Question: What does an assembler do?
Answer: An assembler converts assembly language into machine language [lecture.pdf, page 2].

Context:
{context}

Question: {question}
Answer:"""

def format_context(retrieved):
    return "\n\n".join(
        f"Source filename: {item['metadata']['source']} | pages {item['metadata']['page']}-{item['metadata'].get('page_end', item['metadata']['page'])}\n{item['text']}"
        for item in retrieved
    )

def answer_question(question, k=RETRIEVAL_K, model=LLM_NAME):
    retrieved = retrieve(question, k=k)
    if not retrieved:
        return "I don't know based on the documents.", []
    prompt = PROMPT_TEMPLATE.format(context=format_context(retrieved), question=question)
    response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}], options={"temperature": 0})
    return response["message"]["content"].strip(), retrieved

demo_question = "What is a neural network?"
demo_answer, demo_retrieved = answer_question(demo_question)
print(demo_answer)
print("Retrieved:", [(x["metadata"]["source"], x["metadata"]["page"]) for x in demo_retrieved])


A neural network is a subset of machine learning and is at the heart of deep learning algorithms. Their name and structure are inspired by the human brain, mimicking the way that biological neurons signal to one another. Artificial neural networks (ANNs) are comprised of node layers, containing an input layer, one or more hidden layers, and an output layer. Each node, or artificial neuron, connects to another and has an associated weight and threshold. If the output of any individual node is above the specified threshold value, that node is activated, sending data to the next layer of the network. [Neural lect1.pdf, pages 5-6]
Retrieved: [('Neural lect1.pdf', 10), ('Neural lect1.pdf', 5), ('neural_lab manual.pdf', 6), ('NEURAL NETWORK.pdf', 4), ('Neural lect1.pdf', 12), ('lect6.pdf', 1)]


## 2.6 Evaluation

The evaluation includes questions from neural networks, problem solving, patterns, assembly, and AI. Two questions are intentionally outside the supplied course material. `correct?` is a conservative evidence check: an in-scope answer must contain an expected concept and a source citation, while an out-of-scope answer is correct only when it uses the required uncertainty phrase.

In [5]:
import time

evaluation_questions = [
    {"question": "What is the purpose of an activation function in a neural network?", "terms": ["activation", "neural"], "covered": True},
    {"question": "What is backpropagation used for when training a neural network?", "terms": ["backpropagation", "gradient", "weight"], "covered": True},
    {"question": "What is a heuristic in problem solving?", "terms": ["heuristic", "search"], "covered": True},
    {"question": "What is the aim of the problem-solving course?", "terms": ["aim", "problem-solving"], "anchors": ["aim of this course", "problem solving techniques", "identify several"], "covered": True},
    {"question": "What is pattern recognition?", "terms": ["pattern", "recognition"], "anchors": ["pattern recognition", "distinguish patterns", "learn to distinguish"], "covered": True},
    {"question": "What program converts assembly language to machine language?", "terms": ["assembler", "assembly", "machine"], "anchors": ["assembler", "assembly language to machine language"], "covered": True},
    {"question": "What is the difference between a CPU register and main memory?", "terms": ["register", "memory"], "anchors": ["register", "main memory", "memory operand", "storage"], "covered": True},
    {"question": "What is artificial intelligence trying to achieve?", "terms": ["intelligence", "AI"], "anchors": ["goal", "human intelligence", "machines and software", "decision-making"], "covered": True},
    {"question": "What is the capital of France?", "terms": ["paris"], "covered": False},
    {"question": "What will the weather be tomorrow?", "terms": ["weather"], "covered": False},
]

def judge_from_retrieved_text(answer, retrieved, terms, covered):
    """Make a conservative evidence judgment without tuning questions to pass."""
    answer_lower = answer.lower()
    retrieved_text = " ".join(item["text"] for item in retrieved).lower()
    uncertainty = "i don't know based on the documents" in answer_lower
    has_citation = ("[" in answer and "]" in answer) or any(
        item["metadata"]["source"].lower() in answer_lower for item in retrieved
    )
    if not covered:
        return "Yes" if uncertainty else "No"
    term_supported = any(term.lower() in retrieved_text for term in terms)
    term_answered = any(term.lower() in answer_lower for term in terms)
    return "Yes" if term_supported and term_answered and has_citation and not uncertainty else "No"

def diagnose_failure(item, answer, retrieved):
    """Classify a failed result from evidence visible in its top-k chunks."""
    if not retrieved:
        return "RELEVANCE GUARD", "No chunk passed the cosine-distance threshold; the LLM was not called."
    retrieved_text = " ".join(chunk["text"] for chunk in retrieved).lower()
    anchors = item.get("anchors", item["terms"])
    supported_anchors = [anchor for anchor in anchors if anchor.lower() in retrieved_text]
    if supported_anchors:
        return "GENERATION", f"Direct evidence anchors present in retrieved text: {supported_anchors}"
    return "RETRIEVAL", f"No direct evidence anchors found among: {anchors}."

print(f"\nCosine distance diagnostics (threshold={RELEVANCE_DISTANCE_THRESHOLD})")
for item in evaluation_questions:
    distances = raw_distances(item["question"])
    scope = "in-scope" if item["covered"] else "out-of-scope"
    print(f"  {scope}: {item['question']} -> {[round(distance, 4) for distance in distances]}")

evaluation_rows = []
for item in evaluation_questions:
    answer, retrieved = answer_question(item["question"])
    sources = "; ".join(f"{chunk['metadata']['source']}, p.{chunk['metadata']['page']}" for chunk in retrieved)
    evaluation_rows.append({
        "question": item["question"],
        "retrieved source": sources,
        "answer": answer,
        "correct?": judge_from_retrieved_text(answer, retrieved, item["terms"], item["covered"]),
    })

evaluation_table = pd.DataFrame(evaluation_rows)
pd.set_option("display.max_colwidth", 180)
display(evaluation_table)
in_scope = evaluation_table[[item["covered"] for item in evaluation_questions]]
out_of_scope = evaluation_table[[not item["covered"] for item in evaluation_questions]]
print(f"Honest in-scope score: {(in_scope['correct?'] == 'Yes').sum()}/{len(in_scope)}")
print(f"Out-of-scope refusals: {(out_of_scope['correct?'] == 'Yes').sum()}/{len(out_of_scope)}")

print("\nFailure diagnostics: top-k retrieved chunks and classification")
failed_items = []
for item, row in zip(evaluation_questions, evaluation_rows):
    if row["correct?"] == "Yes":
        continue
    answer, retrieved = answer_question(item["question"])
    category, reason = diagnose_failure(item, answer, retrieved)
    failed_items.append((item, retrieved, category))
    print(f"\nQuestion: {item['question']}\nClassification: {category} ({reason})")
    for rank, chunk in enumerate(retrieved, start=1):
        metadata = chunk["metadata"]
        print(f"\n  Top {rank}: {metadata['source']}, pages {metadata['page']}-{metadata.get('page_end', metadata['page'])}, distance={chunk['distance']:.4f}")
        print(f"  {chunk['text']}")

print(f"\nGeneration-failure retest using configured model: {LLM_NAME}")
for item, retrieved, category in failed_items:
    if category != "GENERATION":
        continue
    retest_answer, _ = answer_question(item["question"], model=LLM_NAME)
    print(f"\n{item['question']}\n{retest_answer}")

print("\nFailure-analysis summary: The corpus audit found no pages containing breadth-first/BFS/queue or the exact phrase design pattern. The stack matches were incidental uses in unrelated topics, so those three original retrieval questions were dropped and replaced with questions grounded in observed page text. The relevance guard uses a 0.62 cosine-distance threshold selected from printed in-scope versus out-of-scope distances; it rejects both out-of-scope queries before the LLM call. The revised prompt explicitly permits partial, cited answers and only allows uncertainty when context is unrelated. Any remaining generation failures are genuine model limitations; the active model is configurable with LLM_MODEL (default llama3.2).")



Cosine distance diagnostics (threshold=0.62)
  in-scope: What is the purpose of an activation function in a neural network? -> [0.3652, 0.4788, 0.4816, 0.4828, 0.484, 0.5047]
  in-scope: What is backpropagation used for when training a neural network? -> [0.34, 0.4103, 0.4152, 0.4195, 0.422, 0.4445]


  in-scope: What is a heuristic in problem solving? -> [0.4432, 0.592, 0.6035, 0.6059, 0.6083, 0.6255]


  in-scope: What is the aim of the problem-solving course? -> [0.3744, 0.6382, 0.6406, 0.6886, 0.69, 0.6981]
  in-scope: What is pattern recognition? -> [0.3008, 0.3008, 0.3247, 0.3247, 0.3577, 0.4254]
  in-scope: What program converts assembly language to machine language? -> [0.3421, 0.3852, 0.4306, 0.4779, 0.5192, 0.5406]
  in-scope: What is the difference between a CPU register and main memory? -> [0.5534, 0.5712, 0.5843, 0.5871, 0.5939, 0.6313]
  in-scope: What is artificial intelligence trying to achieve? -> [0.4134, 0.439, 0.4391, 0.5005, 0.5008, 0.5048]
  out-of-scope: What is the capital of France? -> [0.8559, 0.8581, 0.8581, 0.8887, 0.8888, 0.9056]
  out-of-scope: What will the weather be tomorrow? -> [0.668, 0.706, 0.7224, 0.7425, 0.7702, 0.7711]


,question,retrieved source,answer,correct?
0,What is the purpose of an activation function in a neural network?,"neural_lab manual.pdf, p.1; neural_lab manual.pdf, p.15; NEURAL NETWORK.pdf, p.27; lect4neurall.pdf, p.1; Neural lect1.pdf, p.5; neural_lab manual.pdf, p.12","An activation function in a neural network introduces non-linearity into the model, allowing it to learn more complex relationships between the data than a linear function. Thi...",Yes
1,What is backpropagation used for when training a neural network?,"neural_lab manual.pdf, p.1; lect4neurall.pdf, p.32; NEURAL NETWORK.pdf, p.4; lect6.pdf, p.1; Neural lect1.pdf, p.12; neural_lab manual.pdf, p.35",Backpropagation is used to update the weights of the neurons to minimize the error between the predicted output and the actual output. This process is typically performed using...,Yes
2,What is a heuristic in problem solving?,"L1.pdf, p.1; Introduction to AI.pdf, p.11; Agents in Action.pdf, p.30; Lec 3.1_Dr. Ahmed Elngar.pdf, p.10; L1.pdf, p.5",What is a heuristic in problem solving?\n\nSupports: A heuristic is a technique used in problem solving that involves making educated guesses or using experience to guide the s...,Yes
3,What is the aim of the problem-solving course?,"L1.pdf, p.1","The aim of the problem-solving course is to identify several problem-solving techniques, specifically: recursion, sorting algorithms, searching algorithms, hashing, greedy algo...",Yes
4,What is pattern recognition?,"chapter1 (1).pdf, p.1; chapter1.pdf, p.1; chapter1 (1).pdf, p.4; chapter1.pdf, p.4; Lec 8_Dr. Ahmed Elngar.pdf, p.7; neural_lab manual.pdf, p.58","Pattern recognition is the study of how machines can observe the environment, learn to distinguish patterns of interest, and make sound and reasonable decisions about the categ...",Yes
5,What program converts assembly language to machine language?,"-Mid term-Assembly Language.pdf, p.1; Lecture1.pdf, p.5; Lecture1.pdf, p.1; SHEET1.pdf, p.1; MODEL ANSWER.pdf, p.1; Lec 12_Dr. Ahmed Elngar.pdf, p.25","The assembler converts assembly language into machine language [Lec 12_Dr. Ahmed Elngar.pdf, page 25].",Yes
6,What is the difference between a CPU register and main memory?,"Lecture 2.pdf, p.10; Lecture 2.pdf, p.1; Lecture 2.pdf, p.4; SHEET1.pdf, p.1; -Mid term-Assembly Language.pdf, p.1","The context does not provide a clear answer to this question. The provided documents discuss the 8086 processor, its registers, and instructions, but do not explicitly define o...",No
7,What is artificial intelligence trying to achieve?,"Introduction to AI.pdf, p.10; Introduction to AI.pdf, p.11; Introduction to AI.pdf, p.1; Lec 3.1_Dr. Ahmed Elngar.pdf, p.1; neural_lab manual.pdf, p.6; Introduction to AI.pdf, p.5","Artificial intelligence (AI) is trying to simulate human decision-making, learning, and problem-solving using data and algorithms. Its goal is to create machines and software t...",Yes
8,What is the capital of France?,,I don't know based on the documents.,Yes
9,What will the weather be tomorrow?,,I don't know based on the documents.,Yes


Honest in-scope score: 7/8
Out-of-scope refusals: 2/2

Failure diagnostics: top-k retrieved chunks and classification



Question: What is the difference between a CPU register and main memory?
Classification: GENERATION (Direct evidence anchors present in retrieved text: ['register'])

  Top 1: Lecture 2.pdf, pages 10-15, distance=0.5534
  m <- imm • There is no move mem<-mem instruction. [Page 10] MOVE LIMITATION • Both operand must be in the same size. • There is no instruction to put immediate value directly to segment register. Have to use accumulator (AX) to accomplish this. • T o put immediate value directly to memory, we have to specify its size. (Byte/Word PTR) [Page 11] MOVE (MOV) EXAMPLE •MOV AX,100h •MOV BX,AX •MOV DX,BX •MOV AX,1234h •MOV DX,5678h •MOV AL,DL •MOV BH,DH [Page 12] MOV EXAMPLE •MOV AX,1000h •MOV [100h],AX •MOV BX,[100h] [Page 13] MOV : 16 / 8 BIT REGISTER • T o move value between registers, their size must be the same. [Page 14] EXAMPLE : SHOW MEMORY CONTENTS [Page 15] XCHG : SWAPPING AX = 12h BX = 34h XCHG AX, BX • Same as : MOV DX,AX MOV AX,BX MOV BX,DX AX BX DX

  Top 2: Le


What is the difference between a CPU register and main memory?
The context does not provide a clear answer to this question. The provided documents discuss the 8086 processor, its registers, and instructions, but do not explicitly define or differentiate between CPU registers and main memory.

Failure-analysis summary: The corpus audit found no pages containing breadth-first/BFS/queue or the exact phrase design pattern. The stack matches were incidental uses in unrelated topics, so those three original retrieval questions were dropped and replaced with questions grounded in observed page text. The relevance guard uses a 0.62 cosine-distance threshold selected from printed in-scope versus out-of-scope distances; it rejects both out-of-scope queries before the LLM call. The revised prompt explicitly permits partial, cited answers and only allows uncertainty when context is unrelated. Any remaining generation failures are genuine model limitations; the active model is configurable with L

The corpus audit and failure diagnostics are printed above. The original breadth-first, design-pattern, and assembly-stack questions were removed because the requested topics were not represented as relevant content in the raw corpus; replacement questions use topics found in actual page text. Generation failures are retested with the configured `LLM_MODEL` (default `llama3.2`) using an explicit partial-answer and cited-example prompt.


## 2.7 Export

The configuration is written next to the persistent Chroma collection so the index settings can be reproduced by another script or notebook.

In [6]:
rag_config = {
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "collection_name": COLLECTION_NAME,
    "llm_name": LLM_NAME,
    "llm_model_env": "LLM_MODEL",
    "k": RETRIEVAL_K,
    "relevance_distance_threshold": RELEVANCE_DISTANCE_THRESHOLD,
    "distance": "cosine",
}
config_path = VECTOR_DIR / "rag_config.json"
config_path.write_text(json.dumps(rag_config, indent=2), encoding="utf-8")
print(f"Saved {config_path}")
print(json.dumps(rag_config, indent=2))

Saved C:\rag-assistant-project\backend\data\vector_store\rag_config.json
{
  "chunk_size": 1000,
  "chunk_overlap": 200,
  "embedding_model": "all-MiniLM-L6-v2",
  "collection_name": "course_documents",
  "llm_name": "llama3.2",
  "llm_model_env": "LLM_MODEL",
  "k": 6,
  "relevance_distance_threshold": 0.62,
  "distance": "cosine"
}
